# 01 · Datos y exploración de retornos

Paso 1: descarga de precios diarios **ajustados** (dividendos y splits) de SPY, QQQ, TLT, GLD y EEM, 2005-01-01 a 2025-12-31, con caché local en `data/raw/prices.parquet` (no se sube a Git).

Fuente: Yahoo Finance vía `yfinance` (uso personal/educativo).

In [1]:
import sys; sys.path.append("..")
from src.data import download_prices, to_returns

px = download_prices(cache="../data/raw/prices.parquet")
print(px.shape, px.index.min().date(), px.index.max().date())
print(px.isna().sum())                                  # nulos por ETF
print(px.index.is_monotonic_increasing, px.index.duplicated().sum())
rets = to_returns(px)
print(rets.abs().max())                                 # ningún retorno diario absurdo
print(rets.abs().stack().nlargest(5))                   # los 5 días más extremos

(5282, 5) 2005-01-03 2025-12-30
Ticker
SPY    0
QQQ    0
TLT    0
GLD    0
EEM    0
dtype: int64
True 0
Ticker
SPY    0.145198
QQQ    0.121648
TLT    0.075195
GLD    0.112905
EEM    0.227698
dtype: float64
Date        Ticker
2008-10-13  EEM       0.227698
2008-10-28  EEM       0.209327
2008-10-15  EEM       0.161662
2008-10-13  SPY       0.145198
2008-11-21  EEM       0.144031
dtype: float64


## Notas de calidad de datos (Paso 1)

- **Tamaño:** 5,282 días de negociación × 5 ETFs, del 2005-01-03 al 2025-12-30. **0 nulos**, índice ordenado y sin fechas duplicadas.
- **Fecha final:** `end` en `yfinance` es exclusivo, así que `end="2025-12-31"` termina el 30 de diciembre de 2025. Es una fecha fija: cualquiera obtiene la misma muestra.
- **Días extremos:** los 5 movimientos absolutos más grandes son de la crisis de octubre-noviembre de 2008 (EEM +22.8 % el 2008-10-13; SPY +14.5 % el mismo día). Las peores caídas de SPY, QQQ y TLT son de marzo de 2020 y la de GLD es la del 2013-04-15. Todos coinciden con eventos conocidos: **no hay errores de datos evidentes**.
- **Sesgo de selección del universo:** los 5 ETFs se eligieron hoy, sabiendo que sobrevivieron y son líquidos. Es una limitación real del estudio.
- **Datos:** Yahoo Finance vía `yfinance`, precios ajustados (`auto_adjust=True`), descargados el 2026-11-25. Huella de la caché: `sha256[:12] = 49ae3c57684c`. Yahoo puede recalcular precios ajustados, así que una descarga futura podría diferir un poco. Los datos **no** se suben al repositorio.
